## Demo Notebook

In [1]:
import pandas as pd

from utils.download import *
from pulearn.pubase import *


download_from_gdrive("deceptive-opinion.csv")

documents = pd.read_csv("../data/deceptive-opinion.csv")
documents.head()

[~_o] File already exists --> Aborting download


,deceptive,hotel,polarity,source,text
0,truthful,conrad,positive,TripAdvisor,We stayed for a one night getaway with family ...
1,truthful,hyatt,positive,TripAdvisor,Triple A rate with upgrade to view room was le...
2,truthful,hyatt,positive,TripAdvisor,This comes a little late as I'm finally catchi...
3,truthful,omni,positive,TripAdvisor,The Omni Chicago really delivers on all fronts...
4,truthful,hyatt,positive,TripAdvisor,I asked for a high floor away from the elevato...


Drop not needed columns

In [2]:
documents = documents.drop(columns=["hotel", "source", "polarity"])
documents["deceptive"] = documents["deceptive"].apply(lambda x: 0 if x == "truthful" else 1)
documents.head()

,deceptive,text
0,0,We stayed for a one night getaway with family ...
1,0,Triple A rate with upgrade to view room was le...
2,0,This comes a little late as I'm finally catchi...
3,0,The Omni Chicago really delivers on all fronts...
4,0,I asked for a high floor away from the elevato...


Isolate all deceptive documents and extract a partial test set of 160 instances

In [3]:
deceptive_docs = documents.loc[(documents["deceptive"] == 1)]
deceptive_docs.reset_index(drop=True, inplace=True)
indices = extract_sample(deceptive_docs["deceptive"], ratio=0.2, value=1)
deceptive_set = deceptive_docs.loc[indices]
deceptive_set

,deceptive,text
151,1,We stayed at the Millennium Knickerbocker last...
338,1,The Inter Continental is a great hotel. The st...
645,1,Staying at the Sofitel was one of the less ple...
44,1,"The Omni Chicago is hands down, the best hotel..."
23,1,"The hotel was very nice; Service was great, ev..."
...,...,...
365,1,We really enjoyed our stay at the Palmer House...
129,1,The Swissotel Chicago is a delight to visit. L...
46,1,"Fantastic Hotel! Upscale and luxurious, and in..."
397,1,"Last month, my husband and I stayed at the Int..."


Isolate all non-deceptive documents and extract a partial test set of another 160 instances

In [4]:
non_deceptive_docs = documents.loc[(documents["deceptive"] == 0)]
non_deceptive_docs.reset_index(drop=True, inplace=True)
indices = extract_sample(non_deceptive_docs["deceptive"], ratio=0.2, value=0)
non_deceptive_set = non_deceptive_docs.loc[indices]
non_deceptive_set

,deceptive,text
729,0,"The location is amazing, right across from Nor..."
323,0,My wife and I spent several nights here on a g...
541,0,"Firtly they took $372 off my card, even though..."
477,0,Here are some pros and cons: Pros: -Good locat...
425,0,"Great, awesome service. Seriously, the people ..."
...,...,...
129,0,Having booked the hotel directly with the hote...
211,0,"Very quaint and romantic, yet masculine. I've ..."
646,0,Excellent Hotel ! Rooms and service were great...
608,0,"Not worth the price, even a little bit. A week..."


Merge above sets to create a unified test set

In [5]:
test_set = pd.concat([deceptive_set, non_deceptive_set], ignore_index=True)
test_set

,deceptive,text
0,1,We stayed at the Millennium Knickerbocker last...
1,1,The Inter Continental is a great hotel. The st...
2,1,Staying at the Sofitel was one of the less ple...
3,1,"The Omni Chicago is hands down, the best hotel..."
4,1,"The hotel was very nice; Service was great, ev..."
...,...,...
315,0,Having booked the hotel directly with the hote...
316,0,"Very quaint and romantic, yet masculine. I've ..."
317,0,Excellent Hotel ! Rooms and service were great...
318,0,"Not worth the price, even a little bit. A week..."


Accordingly extract a training set and turn it into a PU dataset

In [6]:
train_set = documents.loc[~(documents["text"].isin(test_set["text"]))]
train_set = train_set.reset_index(drop=True)

indices = extract_sample(train_set["deceptive"], ratio=0.37, value=1)
train_set["pu-label"] = 0
train_set.loc[indices, "pu-label"] = 1

train_set["pu-label"].value_counts()

0    1040
1     237
Name: pu-label, dtype: int64

Define a text preprocessing function

In [7]:
import re

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def prepare(row):
    filtered = []
    row = row.lower().strip()
    row = re.sub('[^A-Za-z0-9.]+', ' ', row)
    row_parts = row.split()
    for part in row_parts:
        filtered.append(lemmatizer.lemmatize(part))
    return " ".join(filtered)

Create TF-IDF vectors

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train = train_set["text"]
X_train = X_train.apply(lambda row: prepare(row))
y_train = train_set["pu-label"]

X_test = test_set["text"]
X_test = X_test.apply(lambda row: prepare(row))
y_test = test_set["deceptive"]

tfidf_input = pd.concat([X_train, X_test])

vectorizer = TfidfVectorizer()

vectors = vectorizer.fit_transform(tfidf_input)
feature_names = vectorizer.get_feature_names_out()
dense = vectors.todense().tolist()

X = pd.DataFrame(dense, columns=feature_names)

train_rows = X_train.shape[0]
test_rows = X_test.shape[0]

X_train = X.iloc[:train_rows]
X_test = X.iloc[-test_rows:]
X_test = X_test.reset_index(drop=True)

Use above vectors to train model

In [9]:
from pulearn.iterative_classifiers import TwostepClassifier
from sklearn.metrics import f1_score


roc_svm = TwostepClassifier("Rocchio", "SVC", params1={"metric": "manhattan"}, params2={"kernel": "linear"})
roc_svm.fit(X_train, y_train)
predictions = roc_svm.predict(X_test)

print("\nF1-score = {:.2f}".format(f1_score(y_test, predictions, average="macro") * 100))

Iteration: 0
Unlabeled points remaining: 398

Iteration: 1
Unlabeled points remaining: 163

Iteration: 2
Unlabeled points remaining: 71

Iteration: 3
Unlabeled points remaining: 21

Iteration: 4
Unlabeled points remaining: 2

Iteration: 5
Unlabeled points remaining: 1

[[  0 243]
 [  1  77]]

F1-score = 68.17
